# Mammo-CLIP Cross-Validation Summary

Run this notebook inside the `mammo_clip` conda environment to aggregate metrics from the training output CSV.



In [9]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, f1_score

CSV_PATH = Path(
    "/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo-clip/outputs/RSNA/zz/Classifier/"
    "upmc_breast_clip_det_b5_period_n_ft/lr_5e-05_epochs_5_weighted_BCE_n_cancer_data_frac_1.0/"
    "seed_10_n_folds_2_outputs.csv"
)

assert CSV_PATH.exists(), f"Missing CSV: {CSV_PATH}"

df = pd.read_csv(CSV_PATH)
df.head()



,patient_id,image_id,cancer,fold,png_path,npz_path,meta_path,label,view,laterality,prediction,prediction_bin
0,DHOWUFOZYUEQ,1962070323431617947923223663535642249,1,0,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,IndexCancer,medio-lateral oblique,R,0.794054,1
1,DHCP7E0MUVIF,110011141431650308222365330479990858383,1,0,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,IndexCancer,cranio-caudal,L,0.896180,1
2,DHDIK5AL9G7R,174372193043778265851095380292359295644,0,0,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,NonCancer,cranio-caudal,R,0.265912,0
3,DHRTEULLIL1K,108071637863790612455404106581422696083,0,0,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,NonCancer,medio-lateral oblique,L,0.551501,1
4,DHC1P3BKX05V,135022485501781938939763389752195957825,0,0,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo...,NonCancer,medio-lateral oblique,L,0.198729,0


In [10]:
def compute_metrics(frame: pd.DataFrame) -> dict:
    y_true = frame["cancer"].astype(int)
    y_prob = frame["prediction"].astype(float)
    y_hat = frame["prediction_bin"].astype(int)

    tn = ((y_true == 0) & (y_hat == 0)).sum()
    fp = ((y_true == 0) & (y_hat == 1)).sum()
    fn = ((y_true == 1) & (y_hat == 0)).sum()
    tp = ((y_true == 1) & (y_hat == 1)).sum()

    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    sens = tp / (tp + fn) if (tp + fn) else float("nan")

    metrics = {
        "auc": roc_auc_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_hat),
        "sensitivity": sens,
        "specificity": spec,
        "f1": f1_score(y_true, y_hat),
        "n": len(frame),
    }

    agg = (
        frame[["patient_id", "laterality", "cancer", "prediction", "prediction_bin"]]
        .groupby(["patient_id", "laterality"], as_index=False)
        .mean()
    )
    agg["prediction_bin"] = (agg["prediction"] >= 0.5).astype(int)

    tn_p = ((agg["cancer"] == 0) & (agg["prediction_bin"] == 0)).sum()
    fp_p = ((agg["cancer"] == 0) & (agg["prediction_bin"] == 1)).sum()
    fn_p = ((agg["cancer"] == 1) & (agg["prediction_bin"] == 0)).sum()
    tp_p = ((agg["cancer"] == 1) & (agg["prediction_bin"] == 1)).sum()

    metrics.update({
        "auc": roc_auc_score(agg["cancer"], agg["prediction"]),
        "accuracy": accuracy_score(agg["cancer"], agg["prediction_bin"]),
        "sensitivity": tp_p / (tp_p + fn_p) if (tp_p + fn_p) else float("nan"),
        "specificity": tn_p / (tn_p + fp_p) if (tn_p + fp_p) else float("nan"),
        "f1": f1_score(agg["cancer"], agg["prediction_bin"]),
        "n": len(agg),
    })

    return metrics

fold_metrics = []
for fold_id in sorted(df["fold"].unique()):
    fold_frame = df[df["fold"] == fold_id]
    metrics = compute_metrics(fold_frame)
    metrics.update({"split": "val", "fold": fold_id})
    fold_metrics.append(metrics)

overall = compute_metrics(df)
overall.update({"split": "val", "fold": "overall"})
fold_metrics.append(overall)

metrics_df = pd.DataFrame(fold_metrics)
metrics_df



,auc,accuracy,sensitivity,specificity,f1,n,split,fold
0,0.846238,0.781640,0.692164,0.863481,0.751773,1122,val,0
1,0.904474,0.840851,0.750274,0.894272,0.777673,7383,val,1
2,0.902891,0.841266,0.750612,0.890822,0.769727,8089,val,overall


In [11]:
display(
    metrics_df.assign(
        auc=lambda d: d["auc"].round(4),
        accuracy=lambda d: d["accuracy"].round(4),
        sensitivity=lambda d: d["sensitivity"].round(4),
        specificity=lambda d: d["specificity"].round(4),
        f1=lambda d: d["f1"].round(4),
    )["fold split auc accuracy sensitivity specificity f1 n".split()]
)



,fold,split,auc,accuracy,sensitivity,specificity,f1,n
0,0,val,0.8462,0.7816,0.6922,0.8635,0.7518,1122
1,1,val,0.9045,0.8409,0.7503,0.8943,0.7777,7383
2,overall,val,0.9029,0.8413,0.7506,0.8908,0.7697,8089


In [12]:
TEST_CSV_PATH = Path(
    "/hpcstor6/scratch01/a/a.kanamarlapudi001/mammo-clip/outputs/RSNA/zz/Classifier/"
    "upmc_breast_clip_det_b5_period_n_ft/lr_5e-05_epochs_5_weighted_BCE_n_cancer_data_frac_1.0/"
    "seed_10_test_eval.csv"
)

test_df = pd.read_csv(TEST_CSV_PATH)
test_metrics = compute_metrics(test_df)
test_metrics.update({"split": "test", "fold": "overall"})



In [13]:
combined = pd.concat([metrics_df, pd.DataFrame([test_metrics])], ignore_index=True)

report = combined.assign(
    auc=lambda d: d["auc"].round(4),
    accuracy=lambda d: d["accuracy"].round(4),
    sensitivity=lambda d: d["sensitivity"].round(4),
    specificity=lambda d: d["specificity"].round(4),
    f1=lambda d: d["f1"].round(4),
)["split fold auc accuracy sensitivity specificity f1 n".split()]

report


,split,fold,auc,accuracy,sensitivity,specificity,f1,n
0,val,0,0.8462,0.7816,0.6922,0.8635,0.7518,1122
1,val,1,0.9045,0.8409,0.7503,0.8943,0.7777,7383
2,val,overall,0.9029,0.8413,0.7506,0.8908,0.7697,8089
3,test,overall,0.8413,0.7612,0.6815,0.8394,0.7387,1135
